### Installation

In [1]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
    !pip install --no-deps unsloth

### Unsloth

In [2]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.4.7: Fast Qwen2 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.97G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/267 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/575 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

In [3]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

### Data Prep


In [4]:
from datasets import load_dataset
dataset = load_dataset("ikram98ai/trademark_detection", split = "train")
test_ds = load_dataset("ikram98ai/trademark_detection", split = "test")

README.md:   0%|          | 0.00/563 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/554k [00:00<?, ?B/s]

val-00000-of-00001.parquet:   0%|          | 0.00/279k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/278k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16521 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/8260 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8261 [00:00<?, ? examples/s]

In [5]:
dataset

Dataset({
    features: ['image_urls', 'trademark_detected', 'organization'],
    num_rows: 16521
})

In [21]:
dataset[2]["image_urls"]

['https://cf.freshprints.com/designs/1722442770557uxzgf_t_front.png']

In [19]:
dataset[2]["trademark_detected"], dataset[2]["organization"]

('No', 'None')

In [5]:
system_prompt = """You are an expert in trademark identification for apparel designs. Your task is to analyze images of apparel and determine
if they contain licensed trademarks such as Greek organization letters (fraternities/sororities) or collegiate/university marks. Your response
must strictly follow this two-line format: first indicating 'Licensed trademarks detected: Yes' or 'Licensed trademarks detected: No', followed
by 'Organization:' with either the specific organization/university name(s) identified or 'None' if no trademarks are detected."""

instruction = """Examine these apparel images and identify if they contain licensed marks or Greek letters. If yes, name the Greek organization or university associated."""


def convert_to_conversation(sample):
    conversation = [
        { "role": "user",
        "content" : [
            {"type" : "text",  "text"  : system_prompt + "\n\n" + instruction},
            ] + [{"type" : "image", "image" : img_url} for img_url in sample["image_urls"]]
        },
        { "role" : "assistant",
        "content" : [
              {"type" : "text",  "text"  : f"Licensed trademarks detected: {sample['trademark_detected']}\nOrganization: {sample['organization']}"} ]
        },
    ]
    return { "messages" : conversation }



Let's convert the dataset into the "correct" format for finetuning:

In [6]:
converted_dataset = [convert_to_conversation(sample) for sample in dataset]

We look at how the conversations are structured for the first example:

In [7]:
converted_dataset[0]

{'messages': [{'role': 'user',
   'content': [{'type': 'text',
     'text': "You are an expert in trademark identification for apparel designs. Your task is to analyze images of apparel and determine\nif they contain licensed trademarks such as Greek organization letters (fraternities/sororities) or collegiate/university marks. Your response\nmust strictly follow this two-line format: first indicating 'Licensed trademarks detected: Yes' or 'Licensed trademarks detected: No', followed\nby 'Organization:' with either the specific organization/university name(s) identified or 'None' if no trademarks are detected.\n\nExamine these apparel images and identify if they contain licensed marks or Greek letters. If yes, name the Greek organization or university associated."},
    {'type': 'image',
     'image': 'https://cf.freshprints.com/designs/1726115067846ljkcb_nt_front.png'},
    {'type': 'image',
     'image': 'https://cf.freshprints.com/designs/1726115067846fvuus_nt_back.png'}]},
  {'role

In [8]:
import requests
from PIL import Image as PILImage
from io import BytesIO

def load_image_from_url(url):
    """Helper function to download and convert image from URL"""
    try:
        response = requests.get(url, stream=True, timeout=10)
        response.raise_for_status()
        return PILImage.open(BytesIO(response.content)).convert("RGB")
    except Exception as e:
        print(f"Error loading image from {url}: {str(e)}")
        return None

Let's first see before we do any finetuning what the model outputs!

In [9]:
idx = 305
test_ds[idx]

{'image_urls': ['https://cf.freshprints.com/designs/1711984624121juwlh_nt_front.png',
  'https://cf.freshprints.com/designs/1711984624121ggdwh_nt_back.png'],
 'trademark_detected': 'Yes',
 'organization': 'Kappa Delta'}

In [10]:
FastVisionModel.for_inference(model) # Enable for inference!

image_urls = dataset[idx]["image_urls"]

messages = [
    {"role": "user", "content": [ {"type": "image"} for url in image_urls ] + [
        {"type": "text", "text": system_prompt + "\n\n" + instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    [load_image_from_url(url) for url in image_urls],
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

Licensed trademarks detected: No  
Organization: None<|im_end|>


### Train the model

In [13]:
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model) # Enable for training!

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer), # Must use!
    train_dataset = converted_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 16,
        gradient_accumulation_steps = 1,
        warmup_steps = 5,
        max_steps = 100,
        # num_train_epochs = 1, # Set this instead of max_steps for full training runs
        learning_rate = 2e-4,
        fp16 = not is_bf16_supported(),
        bf16 = is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",     # For Weights and Biases

        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        dataset_num_proc = 4,
        max_seq_length = 2048,
    ),
)

Unsloth: Model does not have a default image size - using 512


In [14]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA L4. Max memory = 22.161 GB.
7.854 GB of memory reserved.


In [15]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 16,521 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 1 x 1) = 16
 "-____-"     Trainable parameters = 51,521,536/7,000,000,000 (0.74% trained)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss
1,7.300300
2,7.288400
3,7.276700
4,6.834100
5,5.434500
6,3.699800
7,3.019800
8,2.633800
9,2.379600
10,2.099100


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,7.300300
2,7.288400
3,7.276700
4,6.834100
5,5.434500
6,3.699800
7,3.019800
8,2.633800
9,2.379600
10,2.099100


In [16]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

4212.5338 seconds used for training.
70.21 minutes used for training.
Peak reserved memory = 13.58 GB.
Peak reserved memory for training = 5.726 GB.
Peak reserved memory % of max memory = 61.279 %.
Peak reserved memory for training % of max memory = 25.838 %.


### Inference

In [21]:
idx = 3080
dataset[idx]

{'image_urls': ['https://cf.freshprints.com/designs/1730496095991ucduz_nt_front.png'],
 'trademark_detected': 'Yes',
 'organization': 'Alpha Phi'}

In [22]:
FastVisionModel.for_inference(model) # Enable for inference!

image_urls = dataset[idx]["image_urls"]

messages = [
    {"role": "user", "content": [ {"type": "image"} for url in image_urls ] + [
        {"type": "text", "text": system_prompt + "\n\n" + instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    [load_image_from_url(url) for url in image_urls],
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

Licensed trademarks detected: Yes
Organization: Alpha Phi<|im_end|>


### Saving, loading finetuned Lora adapter

In [23]:
model.save_pretrained("trademark_detection_qwen7b_unsloth_lora")
tokenizer.save_pretrained("trademark_detection_qwen7b_unsloth_lora")
model.push_to_hub("ikram98ai/trademark_detection_qwen7b_unsloth_lora")
tokenizer.push_to_hub("ikram98ai/trademark_detection_qwen7b_unsloth_lora")

README.md:   0%|          | 0.00/610 [00:00<?, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/206M [00:00<?, ?B/s]

Saved model to https://huggingface.co/ikram98ai/trademark_detection_qwen7b_unsloth_lora


  0%|          | 0/1 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [47]:
idx = 1000
test_ds[idx]

{'image_urls': ['https://cf.freshprints.com/designs/1725915399446weozx_nt_front.png',
  'https://cf.freshprints.com/designs/1725915399446kgqgq_nt_back.png'],
 'trademark_detected': 'Yes',
 'organization': 'Alpha Phi'}

In [48]:
if False:
    from unsloth import FastVisionModel
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name = "trademark_detection_qwen7b_unsloth_lora",
        load_in_4bit = True, # Set to False for 16bit LoRA
    )
    FastVisionModel.for_inference(model) # Enable for inference!

image_urls = test_ds[idx]["image_urls"]

messages = [
    {"role": "user", "content": [ {"type": "image"} for url in image_urls ] + [
        {"type": "text", "text": system_prompt + "\n\n" + instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    [load_image_from_url(url) for url in image_urls],
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

Licensed trademarks detected: Yes
Organization: Alpha Phi<|im_end|>
